# Notebook 1: Analyse Exploratoire des Données (EDA)

## Credit Scoring avec IA Explicable (XAI)

### Objectifs
- Charger et explorer le German Credit Dataset
- Analyser la distribution des variables
- Identifier les corrélations
- Détecter les valeurs manquantes et aberrantes
- Analyser le déséquilibre de classe

In [ ]:
# Import des bibliothèques
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import des modules du projet
from src.data_loader import load_data
from src.config import NUMERICAL_COLUMNS, CATEGORICAL_COLUMNS

print("✅ Import des bibliothèques réussi")

## 1. Chargement des Données

In [ ]:
# Charger les données
X, y = load_data()

print(f"✅ Données chargées avec succès")
print(f"   - Shape de X: {X.shape}")
print(f"   - Shape de y: {y.shape}")

In [ ]:
# Afficher les premières lignes
print("\n📊 Premières lignes du dataset:")
display(X.head())

In [ ]:
# Informations sur les types de données
print("\n📋 Types de données:")
print(X.dtypes)

In [ ]:
# Statistiques descriptives
print("\n📈 Statistiques descriptives:")
display(X.describe())

## 2. Analyse de la Variable Cible

In [ ]:
# Distribution de la cible
target_counts = y.value_counts()
target_percentages = y.value_counts(normalize=True) * 100

print("\n🎯 Distribution de la cible:")
print(f"   - Bad Credit (0): {target_counts[0]} ({target_percentages[0]:.1f}%)")
print(f"   - Good Credit (1): {target_counts[1]} ({target_percentages[1]:.1f}%)")

In [ ]:
# Visualisation de la distribution de la cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
sns.countplot(x=y, ax=axes[0])
axes[0].set_title('Distribution du Risque de Crédit')
axes[0].set_xlabel('Risque de Crédit')
axes[0].set_ylabel('Nombre d\'instances')
axes[0].set_xticklabels(['Bad Credit', 'Good Credit'])

# Pie chart
axes[1].pie(target_counts, labels=['Bad Credit', 'Good Credit'], autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Répartition du Risque de Crédit')

plt.tight_layout()
plt.show()

print("\n⚠️  Observation: Déséquilibre de classe modéré (70/30)")

## 3. Analyse des Variables Numériques

In [ ]:
# Sélectionner les variables numériques
numerical_df = X[NUMERICAL_COLUMNS].copy()

print(f"\n📊 Variables numériques: {len(NUMERICAL_COLUMNS)}")
print(NUMERICAL_COLUMNS)

In [ ]:
# Histogrammes des variables numériques
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLUMNS):
    sns.histplot(data=X, x=col, kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution de {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Fréquence')

# Supprimer les axes vides
for i in range(len(NUMERICAL_COLUMNS), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots des variables numériques
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLUMNS):
    sns.boxplot(data=X, y=col, ax=axes[i])
    axes[i].set_title(f'Boxplot de {col}')
    axes[i].set_ylabel(col)

# Supprimer les axes vides
for i in range(len(NUMERICAL_COLUMNS), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des variables numériques par classe
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLUMNS):
    sns.boxplot(data=X.assign(credit_risk=y), x='credit_risk', y=col, ax=axes[i])
    axes[i].set_title(f'{col} par Risque de Crédit')
    axes[i].set_xlabel('Risque de Crédit')
    axes[i].set_ylabel(col)
    axes[i].set_xticklabels(['Bad', 'Good'])

# Supprimer les axes vides
for i in range(len(NUMERICAL_COLUMNS), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

## 4. Analyse des Variables Catégorielles

In [ ]:
# Sélectionner les variables catégorielles
categorical_df = X[CATEGORICAL_COLUMNS].copy()

print(f"\n📊 Variables catégorielles: {len(CATEGORICAL_COLUMNS)}")
print(CATEGORICAL_COLUMNS)

In [ ]:
# Distribution des variables catégorielles
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(CATEGORICAL_COLUMNS):
    value_counts = X[col].value_counts()
    axes[i].bar(range(len(value_counts)), value_counts.values)
    axes[i].set_title(f'Distribution de {col}')
    axes[i].set_xticks(range(len(value_counts)))
    axes[i].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[i].set_ylabel('Nombre')

# Supprimer les axes vides
for i in range(len(CATEGORICAL_COLUMNS), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des variables catégorielles par classe
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(CATEGORICAL_COLUMNS):
    # Créer un tableau croisé
    crosstab = pd.crosstab(X[col], y, normalize='index') * 100
    crosstab.plot(kind='bar', ax=axes[i], stacked=True)
    axes[i].set_title(f'{col} par Risque de Crédit')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Pourcentage')
    axes[i].legend(['Bad', 'Good'], loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)

# Supprimer les axes vides
for i in range(len(CATEGORICAL_COLUMNS), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

## 5. Analyse des Corrélations

In [ ]:
# Matrice de corrélation des variables numériques
correlation_matrix = X[NUMERICAL_COLUMNS].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation des Variables Numériques')
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations les plus fortes
corr_pairs = correlation_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs != 1.0].abs().sort_values(ascending=False)

print("\n🔗 Top 10 des corrélations les plus fortes:")
for i, (idx, value) in enumerate(corr_pairs.head(10).items(), 1):
    print(f"   {i}. {idx[0]} - {idx[1]}: {value:.3f}")

In [ ]:
# Scatter plots des corrélations les plus fortes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Duration vs Credit Amount
sns.scatterplot(data=X, x='duration', y='credit_amount', hue=y, ax=axes[0])
axes[0].set_title('Duration vs Credit Amount')
axes[0].set_xlabel('Duration (mois)')
axes[0].set_ylabel('Credit Amount (DM)')
axes[0].legend(['Bad', 'Good'])

# Age vs Credit Amount
sns.scatterplot(data=X, x='age', y='credit_amount', hue=y, ax=axes[1])
axes[1].set_title('Age vs Credit Amount')
axes[1].set_xlabel('Age (années)')
axes[1].set_ylabel('Credit Amount (DM)')
axes[1].legend(['Bad', 'Good'])

# Duration vs Age
sns.scatterplot(data=X, x='duration', y='age', hue=y, ax=axes[2])
axes[2].set_title('Duration vs Age')
axes[2].set_xlabel('Duration (mois)')
axes[2].set_ylabel('Age (années)')
axes[2].legend(['Bad', 'Good'])

# Installment Rate vs Credit Amount
sns.scatterplot(data=X, x='installment_rate', y='credit_amount', hue=y, ax=axes[3])
axes[3].set_title('Installment Rate vs Credit Amount')
axes[3].set_xlabel('Installment Rate (%)')
axes[3].set_ylabel('Credit Amount (DM)')
axes[3].legend(['Bad', 'Good'])

plt.tight_layout()
plt.show()

## 6. Analyse des Features Sensibles

In [ ]:
# Distribution par genre
gender_counts = X['gender'].value_counts()
print("\n👤 Distribution par genre:")
print(gender_counts)

# Distribution par groupe d'âge
age_group_counts = X['age_group'].value_counts()
print("\n📅 Distribution par groupe d'âge:")
print(age_group_counts)

In [ ]:
# Distribution du risque de crédit par genre
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genre
sns.countplot(data=X.assign(credit_risk=y), x='gender', hue='credit_risk', ax=axes[0])
axes[0].set_title('Risque de Crédit par Genre')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Nombre d\'instances')
axes[0].legend(['Bad', 'Good'])

# Groupe d'âge
sns.countplot(data=X.assign(credit_risk=y), x='age_group', hue='credit_risk', ax=axes[1])
axes[1].set_title('Risque de Crédit par Groupe d\'Âge')
axes[1].set_xlabel('Groupe d\'Âge')
axes[1].set_ylabel('Nombre d\'instances')
axes[1].legend(['Bad', 'Good'])

plt.tight_layout()
plt.show()

In [ ]:
# Taux de bon crédit par genre et groupe d'âge
gender_good_rate = X.assign(credit_risk=y).groupby('gender')['credit_risk'].mean() * 100
age_group_good_rate = X.assign(credit_risk=y).groupby('age_group')['credit_risk'].mean() * 100

print("\n📊 Taux de bon crédit par genre:")
for gender, rate in gender_good_rate.items():
    print(f"   - {gender}: {rate:.1f}%")

print("\n📊 Taux de bon crédit par groupe d'âge:")
for age_group, rate in age_group_good_rate.items():
    print(f"   - {age_group}: {rate:.1f}%")

## 7. Valeurs Manquantes et Aberrantes

In [ ]:
# Vérification des valeurs manquantes
missing_values = X.isnull().sum()
missing_percentage = (X.isnull().sum() / len(X)) * 100

print("\n🔍 Valeurs manquantes:")
if missing_values.sum() == 0:
    print("   ✅ Aucune valeur manquante détectée")
else:
    missing_df = pd.DataFrame({
        'Valeurs manquantes': missing_values,
    'Pourcentage': missing_percentage
    })
    missing_df = missing_df[missing_df['Valeurs manquantes'] > 0]
    display(missing_df)

In [ ]:
# Détection des valeurs aberrantes (IQR method)
print("\n⚠️  Valeurs aberrantes détectées (méthode IQR):")

for col in NUMERICAL_COLUMNS:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = X[(X[col] < lower_bound) | (X[col] > upper_bound)]
    
    if len(outliers) > 0:
        print(f"   - {col}: {len(outliers)} valeurs aberrantes ({len(outliers)/len(X)*100:.1f}%)")

## 8. Résumé de l'EDA

In [ ]:
print("\n" + "="*60)
print("📊 RÉSUMÉ DE L'ANALYSE EXPLORATOIRE DES DONNÉES")
print("="*60)

print("\n📋 Caractéristiques du dataset:")
print(f"   - Nombre d'instances: {len(X)}")
print(f"   - Nombre de features: {X.shape[1]}")
print(f"   - Features numériques: {len(NUMERICAL_COLUMNS)}")
print(f"   - Features catégorielles: {len(CATEGORICAL_COLUMNS)}")

print("\n🎯 Distribution de la cible:")
print(f"   - Bad Credit: {target_counts[0]} ({target_percentages[0]:.1f}%)")
print(f"   - Good Credit: {target_counts[1]} ({target_percentages[1]:.1f}%)")
print(f"   - Ratio: {target_counts[1]/target_counts[0]:.2f}")

print("\n🔗 Corrélations principales:")
for i, (idx, value) in enumerate(corr_pairs.head(5).items(), 1):
    print(f"   {i}. {idx[0]} - {idx[1]}: {value:.3f}")

print("\n👤 Distribution par genre:")
for gender, count in gender_counts.items():
    print(f"   - {gender}: {count} ({count/len(X)*100:.1f}%)")

print("\n📅 Distribution par groupe d'âge:")
for age_group, count in age_group_counts.items():
    print(f"   - {age_group}: {count} ({count/len(X)*100:.1f}%)")

print("\n✅ Qualité des données:")
print(f"   - Valeurs manquantes: {missing_values.sum()}")
print(f"   - Types de données: {X.dtypes.nunique()} types différents")

print("\n" + "="*60)
print("✅ Analyse exploratoire terminée avec succès")
print("="*60)

## 9. Insights Clés

### Points Importants:

1. **Déséquilibre de classe**: 70% de bons crédits vs 30% de mauvais crédits
2. **Corrélation forte**: Durée et montant du crédit sont positivement corrélés
3. **Distribution asymétrique**: La plupart des variables numériques ont des distributions asymétriques
4. **Pas de valeurs manquantes**: Le dataset est complet
5. **Disparités de genre**: Légère différence dans les taux de bon crédit par genre
6. **Impact de l'âge**: Les jeunes emprunteurs ont un taux de bon crédit plus faible

### Recommandations:

1. Utiliser des techniques de gestion du déséquilibre de classe (class_weight, SMOTE)
2. Considérer la transformation logarithmique pour les variables asymétriques
3. Surveiller les métriques de fairness par genre et âge
4. Explorer les interactions entre durée et montant du crédit